# **Pandas en Python:**
> **Nota:** Este notebook está hecho en **Polars** para que puedas comparar **sección por sección** con tu notebook original de Pandas.

## Introducción y explicación en profundidad de **Polars** (y comparación con Pandas)

**Polars** es una librería de DataFrames para Python diseñada para ser **muy rápida** y **eficiente en memoria**. Su núcleo está implementado en **Rust** y muchas operaciones usan un modelo **columnar** (alineado con **Apache Arrow**), lo que suele acelerar transformaciones por columnas.

### Por qué Polars suele ser más rápido
- **Expresiones (vectorización):** transformaciones declarativas con `pl.col(...)`, `pl.when(...)`, etc.
- **Multihilo:** muchas operaciones se paralelizan automáticamente.
- **Modo Lazy (planificador + optimizador):** construye un plan y lo optimiza antes de ejecutar:
  - *Predicate pushdown* (empuja filtros hacia la lectura),
  - *Projection pushdown* (lee solo columnas necesarias),
  - reordenación de operaciones para reducir coste.

### Diferencias importantes con Pandas
- **No hay `Index` como concepto central.** En Polars normalmente mantienes `id` como columna.
- **API distinta:** Polars se parece más a SQL (expresiones) que a Pandas (estilo imperativo).
- **UDFs tipo `apply`** existen, pero se recomiendan evitar en datos grandes (penalizan rendimiento/optimizaciones).

### Cuándo usar cada uno
- **Polars:** datos grandes, pipelines largos, rendimiento.
- **Pandas:** ecosistema enorme, compatibilidad, exploración rápida.
- Mezcla típica: transformar con Polars y, si hace falta, `df.to_pandas()`.


### **Importar librería**


In [1]:
# En Colab, instala Polars si hace falta:
!pip -q install polars

import polars as pl
import random


In [ ]:
# (Celda vacía en el original)


In [2]:
# (Celda vacía en el original)
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


### **Carga de Datos**


In [3]:
# CARGA DE DATOS
df = pl.read_csv("/content/drive/MyDrive/CARLOS III 2023 2024/Colab Notebooks/Sistemas Big Data/dataset.csv") # Por defecto detecta las columnas por el encabezado del archivo
# Puedo especificar algunos parametros adicionales como por ejemplo una columna como indice
# pl.read_csv("./dataset.csv", index_col = "id")

df.head()


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183721,"""Flying home to run down from t…",23,null,10,"""ECUADOR""","""leonardokuffo""",389,258
183722,"""Today we commemorate and MNML …",500,21,null,"""BRASIL""","""mateusmartins""",982,1822
183723,"""Today we have reached US$6.55 …",190,123,6,"""MEXICO""","""pedrojuarez""",12,129
183724,"""Faking It by Joel Atwell. Writ…",131,76,3,"""ECUADOR""","""galocastillo""",332,378
183725,"""Welcome back! 🙌""",113,130,9,"""MEXICO""","""pedrojuarez""",12,129


### **Asignar la columna Id como índice**
> **Diferencia:** en Polars no existe `Index` como en Pandas, por eso `index_col="id"` no aplica.  
> Equivalente práctico: mantener `id` como columna y **ordenar** por `id` (o filtrar por `id`).


In [4]:
# (LÍNEA ORIGINAL EN PANDAS PARA COMPARAR)
# # Puedo especificar algunos parametros adicionales como por ejemplo una columna como indice
# df =pd.read_csv(("/content/drive/MyDrive/CARLOS III 2023 2024/Colab Notebooks/Sistemas Big Data/dataset.csv"), index_col = "id")

# En Polars no existe index_col/Index. Leemos y ordenamos por 'id' (equivalente práctico).
df = pl.read_csv(("/content/drive/MyDrive/CARLOS III 2023 2024/Colab Notebooks/Sistemas Big Data/dataset.csv"))
df = df.sort('id')
df.head()


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183721,"""Flying home to run down from t…",23,null,10,"""ECUADOR""","""leonardokuffo""",389,258
183722,"""Today we commemorate and MNML …",500,21,null,"""BRASIL""","""mateusmartins""",982,1822
183723,"""Today we have reached US$6.55 …",190,123,6,"""MEXICO""","""pedrojuarez""",12,129
183724,"""Faking It by Joel Atwell. Writ…",131,76,3,"""ECUADOR""","""galocastillo""",332,378
183725,"""Welcome back! 🙌""",113,130,9,"""MEXICO""","""pedrojuarez""",12,129


### **Mostrar la cabecera y primeras 8 filas**


In [5]:
df.head(8)


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183721,"""Flying home to run down from t…",23,null,10,"""ECUADOR""","""leonardokuffo""",389,258
183722,"""Today we commemorate and MNML …",500,21,null,"""BRASIL""","""mateusmartins""",982,1822
183723,"""Today we have reached US$6.55 …",190,123,6,"""MEXICO""","""pedrojuarez""",12,129
183724,"""Faking It by Joel Atwell. Writ…",131,76,3,"""ECUADOR""","""galocastillo""",332,378
183725,"""Welcome back! 🙌""",113,130,9,"""MEXICO""","""pedrojuarez""",12,129
183726,"""Contest: Win a fan of his ass.…",492,70,6,"""BRASIL""","""mateusmartins""",982,1822
183727,"""80's & friends! ✈️""",158,40,22,"""ECUADOR""","""leonardokuffo""",389,258
183728,"""Thank you guess how did I feel…",null,50,10,"""MEXICO""","""pedrojuarez""",12,129


### **Mostrar las ultimas 8 filas**


In [6]:
df.tail(8)


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183740,"""Programming is a hot topic!""",133,145,15,"""BRASIL""","""lucasperes""",82,351
183741,"""Programming? i love it!""",92,146,1,"""ECUADOR""","""galocastillo""",332,378
183742,"""WHAT???""",255,73,17,"""ECUADOR""","""galocastillo""",332,378
183743,"""Amazing video by Leonardo!""",432,95,18,"""BRASIL""","""lucasperes""",82,351
183744,"""Thanks man!""",430,143,28,"""BRASIL""","""lucasperes""",null,351
183745,"""There is nothing better than p…",424,110,29,"""BRASIL""","""lucasperes""",82,351
183746,"""BORED AF""",488,28,27,"""MEXICO""","""gabrielcarvajal""",21,2721
183747,"""I do not know if i like progra…",318,58,20,"""BRASIL""","""isabelladasilva""",928,9918


### **Proporciona Estadísticas Descriptivas**


In [7]:
df.describe()


statistic,id,full_text,favorites,retweets,mentions,country,user,followers,followees
str,f64,str,f64,f64,f64,str,str,f64,f64
"""count""",27.0,"""27""",26.0,26.0,26.0,"""27""","""27""",26.0,27.0
"""null_count""",0.0,"""0""",1.0,1.0,1.0,"""0""","""0""",1.0,0.0
"""mean""",183734.0,null,280.538462,80.0,15.423077,null,null,352.807692,1190.185185
"""std""",7.937254,null,153.377242,40.303846,9.596554,null,null,375.319493,1965.735995
"""min""",183721.0,"""80's & friends! ✈️""",23.0,21.0,1.0,"""BRASIL""","""gabrielcarvajal""",12.0,129.0
"""25%""",183728.0,null,133.0,44.0,8.0,null,null,21.0,258.0
"""50%""",183734.0,null,315.0,82.0,17.0,null,null,332.0,351.0
"""75%""",183741.0,null,424.0,111.0,24.0,null,null,389.0,1822.0
"""max""",183747.0,"""not feeling god right now""",500.0,146.0,29.0,"""MEXICO""","""pedrojuarez""",982.0,9918.0


### **Proporciona Estadísticas sobre columnas categóricas o de texto**
> En Pandas: `df.describe(include='all')`.  
> En Polars construimos un resumen útil: `dtype`, `nulls`, `n_unique`, y un top frecuente por columna categórica.


In [8]:
# Resumen general por columna
resumen = []
for c in df.columns:
    s = df.get_column(c)
    resumen.append({
        "columna": c,
        "dtype": str(s.dtype),
        "nulls": int(s.null_count()),
        "n_unique": int(s.n_unique()),
    })
pl.DataFrame(resumen)


columna,dtype,nulls,n_unique
str,str,i64,i64
"""id""","""Int64""",0,27
"""full_text""","""String""",0,27
"""favorites""","""Int64""",1,27
"""retweets""","""Int64""",1,25
"""mentions""","""Int64""",1,19
"""country""","""String""",0,3
"""user""","""String""",0,7
"""followers""","""Int64""",1,8
"""followees""","""Int64""",0,7


In [9]:
# Top 5 valores más frecuentes de una columna categórica (ej: country)
df.group_by("country").len().sort("len", descending=True).head(5)


country,len
str,u32
"""BRASIL""",11
"""MEXICO""",8
"""ECUADOR""",8


### **Proporciona los nombres de todas las columnas en el DataFrame como un objeto Index de pandas.**


In [10]:
df.columns


['id',
 'full_text',
 'favorites',
 'retweets',
 'mentions',
 'country',
 'user',
 'followers',
 'followees']

#### **Quitar filas con elemenos NaN.**


In [11]:
# En Polars: valores ausentes suelen ser null
df_filtrado = df.drop_nulls()
df_filtrado.head()


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183723,"""Today we have reached US$6.55 …",190,123,6,"""MEXICO""","""pedrojuarez""",12,129
183724,"""Faking It by Joel Atwell. Writ…",131,76,3,"""ECUADOR""","""galocastillo""",332,378
183725,"""Welcome back! 🙌""",113,130,9,"""MEXICO""","""pedrojuarez""",12,129
183726,"""Contest: Win a fan of his ass.…",492,70,6,"""BRASIL""","""mateusmartins""",982,1822
183727,"""80's & friends! ✈️""",158,40,22,"""ECUADOR""","""leonardokuffo""",389,258


#### **Llenar los valores NaN con un valor por defecto.**


In [12]:
df_filtrado_con_valores_por_defecto = df.fill_null(0)
df_filtrado_con_valores_por_defecto.head()


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183721,"""Flying home to run down from t…",23,0,10,"""ECUADOR""","""leonardokuffo""",389,258
183722,"""Today we commemorate and MNML …",500,21,0,"""BRASIL""","""mateusmartins""",982,1822
183723,"""Today we have reached US$6.55 …",190,123,6,"""MEXICO""","""pedrojuarez""",12,129
183724,"""Faking It by Joel Atwell. Writ…",131,76,3,"""ECUADOR""","""galocastillo""",332,378
183725,"""Welcome back! 🙌""",113,130,9,"""MEXICO""","""pedrojuarez""",12,129


#### **Llenar los valores de una columna que contenga NaN con un valor por defecto.**


In [13]:
df_filtrado_con_valores_por_defecto_en_columna = df.with_columns([
    pl.col("retweets").fill_null(0),
    pl.col("mentions").fill_null(-1),
    pl.col("favorites").fill_null(0),
])
df_filtrado_con_valores_por_defecto_en_columna.head()


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183721,"""Flying home to run down from t…",23,0,10,"""ECUADOR""","""leonardokuffo""",389,258
183722,"""Today we commemorate and MNML …",500,21,-1,"""BRASIL""","""mateusmartins""",982,1822
183723,"""Today we have reached US$6.55 …",190,123,6,"""MEXICO""","""pedrojuarez""",12,129
183724,"""Faking It by Joel Atwell. Writ…",131,76,3,"""ECUADOR""","""galocastillo""",332,378
183725,"""Welcome back! 🙌""",113,130,9,"""MEXICO""","""pedrojuarez""",12,129


### **Muestra el tipo de dato de cada columna en el DataFrame**


In [14]:
df_filtrado_con_valores_por_defecto_en_columna.schema


Schema([('id', Int64),
        ('full_text', String),
        ('favorites', Int64),
        ('retweets', Int64),
        ('mentions', Int64),
        ('country', String),
        ('user', String),
        ('followers', Int64),
        ('followees', Int64)])

### **Mostrar una columna o mas del dataframe**


In [15]:
df["favorites"].head(5)
df.select(["favorites","country"]).head(5)


favorites,country
i64,str
23,"""ECUADOR"""
500,"""BRASIL"""
190,"""MEXICO"""
131,"""ECUADOR"""
113,"""MEXICO"""


# **Filtrado**


### **Dar las ultimas 5**


In [ ]:
df.tail(5)


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183743,"""Amazing video by Leonardo!""",432,95,18,"""BRASIL""","""lucasperes""",82,351
183744,"""Thanks man!""",430,143,28,"""BRASIL""","""lucasperes""",null,351
183745,"""There is nothing better than p…",424,110,29,"""BRASIL""","""lucasperes""",82,351
183746,"""BORED AF""",488,28,27,"""MEXICO""","""gabrielcarvajal""",21,2721
183747,"""I do not know if i like progra…",318,58,20,"""BRASIL""","""isabelladasilva""",928,9918


### **Dar las primeras 5**


In [ ]:
df.head(5)


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183721,"""Flying home to run down from t…",23,null,10,"""ECUADOR""","""leonardokuffo""",389,258
183722,"""Today we commemorate and MNML …",500,21,null,"""BRASIL""","""mateusmartins""",982,1822
183723,"""Today we have reached US$6.55 …",190,123,6,"""MEXICO""","""pedrojuarez""",12,129
183724,"""Faking It by Joel Atwell. Writ…",131,76,3,"""ECUADOR""","""galocastillo""",332,378
183725,"""Welcome back! 🙌""",113,130,9,"""MEXICO""","""pedrojuarez""",12,129


### **Dar la primera fila**


In [ ]:
df.head(1)


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183721,"""Flying home to run down from t…",23,null,10,"""ECUADOR""","""leonardokuffo""",389,258


#### **Filtrar por los Identificadores**


In [ ]:
df.filter(pl.col("id").is_in([183743, 183744])).select(["favorites","country","id"])


favorites,country,id
i64,str,i64
432,"""BRASIL""",183743
430,"""BRASIL""",183744


## **Filtrado con Condiciones**


#### Obtener los registros donde los favoritos son mayor que 400


In [ ]:
df.filter(pl.col("favorites").fill_null(0) > 400)


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183722,"""Today we commemorate and MNML …",500,21,null,"""BRASIL""","""mateusmartins""",982,1822
183726,"""Contest: Win a fan of his ass.…",492,70,6,"""BRASIL""","""mateusmartins""",982,1822
183733,"""Programming is the best!""",467,69,10,"""ECUADOR""","""leonardokuffo""",389,258
183735,"""Buy this product NOW!!""",418,24,2,"""MEXICO""","""gabrielcarvajal""",21,2721
183743,"""Amazing video by Leonardo!""",432,95,18,"""BRASIL""","""lucasperes""",82,351
183744,"""Thanks man!""",430,143,28,"""BRASIL""","""lucasperes""",null,351
183745,"""There is nothing better than p…",424,110,29,"""BRASIL""","""lucasperes""",82,351
183746,"""BORED AF""",488,28,27,"""MEXICO""","""gabrielcarvajal""",21,2721


#### Obtener los registros donde los favoritos son mayor que 400 y metions sea mayor que 20


In [ ]:
df.filter(
    (pl.col("favorites").fill_null(0) > 400) &
    (pl.col("mentions").fill_null(-1) > 20)
)


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183744,"""Thanks man!""",430,143,28,"""BRASIL""","""lucasperes""",null,351
183745,"""There is nothing better than p…",424,110,29,"""BRASIL""","""lucasperes""",82,351
183746,"""BORED AF""",488,28,27,"""MEXICO""","""gabrielcarvajal""",21,2721


#### Obtener los registros que tenga en full_text "Programming"


In [ ]:
df.filter(pl.col("full_text").str.contains("Programming"))


id,full_text,favorites,retweets,mentions,country,user,followers,followees
i64,str,i64,i64,i64,str,str,i64,i64
183733,"""Programming is the best!""",467,69,10,"""ECUADOR""","""leonardokuffo""",389,258
183740,"""Programming is a hot topic!""",133,145,15,"""BRASIL""","""lucasperes""",82,351
183741,"""Programming? i love it!""",92,146,1,"""ECUADOR""","""galocastillo""",332,378


# **Transformación de Datos**
Apartir de los datos se puede hacer limpieza, transformar o calcular nuevas columnas.
> Diferencia: Polars recomienda expresiones vectorizadas. UDFs tipo `apply` (aquí `map_elements`) pueden ser más lentas.


#### Transformacion de datos de una columna
Rellenar `retweets` con 0 y `mentions` con -1, y crear `ganancias`.


In [ ]:
df = df.with_columns([
    pl.col("retweets").fill_null(0),
    pl.col("mentions").fill_null(-1),
    pl.col("favorites").fill_null(0),
])

df = df.with_columns(
    pl.col("retweets")
      .map_elements(lambda x: int(x) * random.randint(3,5), return_dtype=pl.Int64)
      .alias("ganancias")
)

df.select(["id","retweets","ganancias","mentions"]).head(10)


id,retweets,ganancias,mentions
i64,i64,i64,i64
183721,0,0,10
183722,21,63,-1
183723,123,369,6
183724,76,228,3
183725,130,520,9
183726,70,280,6
183727,40,200,22
183728,50,150,10
183729,82,328,26


#### Transformación de datos de dos o mas columnas
`popularidad = followees / followers` (con protección si followers==0).


In [ ]:
df = df.with_columns(
    pl.when(pl.col("followers") == 0)
      .then(None)
      .otherwise(pl.col("followees") / pl.col("followers"))
      .alias("popularidad")
)

df.select(["id","followees","followers","popularidad"]).head(10)


id,followees,followers,popularidad
i64,i64,i64,f64
183721,258,389,0.663239
183722,1822,982,1.855397
183723,129,12,10.75
183724,378,332,1.138554
183725,129,12,10.75
183726,1822,982,1.855397
183727,258,389,0.663239
183728,129,12,10.75
183729,129,12,10.75


### **Transformación agrupación o agregación**


#### Sumar la información de otras columnas mediante la agre...to calcular el promedio de "Me gustas" en los tweet por páis.


In [ ]:
df_mean = (
    df.select(["country","favorites","retweets","mentions","followers","followees","ganancias","popularidad"])
      .group_by("country")
      .mean()
)

df_mean


country,favorites,retweets,mentions,followers,followees,ganancias,popularidad
str,f64,f64,f64,f64,f64,f64,f64
"""BRASIL""",339.909091,91.272727,17.727273,616.6,1889.363636,371.181818,3.708644
"""ECUADOR""",186.375,56.75,11.875,360.5,318.0,240.5,0.900897
"""MEXICO""",258.0,77.75,13.75,15.375,1101.0,295.875,55.308036


#### Agrupando por pais y aplicando diferentes funciones de ...lumna (followers: suma , mentions: media y retweets: maximo )


In [ ]:
grouped = (
    df.group_by("country")
      .agg([
          pl.col("followers").sum().alias("followers"),
          pl.col("mentions").mean().alias("mentions"),
          pl.col("retweets").max().alias("retweets"),
      ])
)

grouped


country,followers,mentions,retweets
str,i64,f64,i64
"""MEXICO""",123,13.75,130
"""BRASIL""",6166,17.727273,145
"""ECUADOR""",2884,11.875,146


#### Crear un DataFrame llamado "grouped" que contenga la agrupación anterior


In [ ]:
grouped.head()


country,followers,mentions,retweets
str,i64,f64,i64
"""MEXICO""",123,13.75,130
"""BRASIL""",6166,17.727273,145
"""ECUADOR""",2884,11.875,146


#### Filtrar los follower de DataFrame anterior donde sean mayores de 5000


In [ ]:
grouped.filter(pl.col("followers") > 5000)


country,followers,mentions,retweets
str,i64,f64,i64
"""BRASIL""",6166,17.727273,145


In [ ]:
# (Celda vacía en el original)


#### Guardar nuestros datos transformados a CSV, JSON, SQL, ...


In [ ]:
grouped.write_csv("/content/drive/MyDrive/CARLOS III 2023 2024/Colab Notebooks/Sistemas Big Data/grouped.csv")
grouped.write_ndjson("/content/drive/MyDrive/CARLOS III 2023 2024/Colab Notebooks/Sistemas Big Data/grouped.ndjson")


---
## Extra (opcional): versión Lazy equivalente (esto es “extra” respecto a Pandas)


In [ ]:
lf = pl.scan_csv("/content/drive/MyDrive/CARLOS III 2023 2024/Colab Notebooks/Sistemas Big Data/dataset.csv")

resultado_lazy = (
    lf.with_columns([
        pl.col("favorites").fill_null(0),
        pl.col("retweets").fill_null(0),
        pl.col("mentions").fill_null(-1),
    ])
    .filter(pl.col("favorites") > 400)
    .group_by("country")
    .agg(pl.col("followers").sum().alias("followers"))
    .sort("followers", descending=True)
    .collect()
)

resultado_lazy


country,followers
str,i64
"""BRASIL""",2128
"""ECUADOR""",389
"""MEXICO""",42
